In [0]:
%run ./01_setup_paths_and_schema

In [0]:
dbutils.fs.rm(
    f"{bronze_path}/patients",
    True
)

In [0]:


from pyspark.sql.functions import *

df = spark.read.format("delta") \
.load(f"{bronze_path}/patients")

expected = StructType(

patients_schema.fields +

[

StructField(
"ingestion_time",
TimestampType(),
True
)

]

)

missing,datatype,extra = validate_schema(

df,

expected

)

print("Missing =",missing)

print("Datatype =",datatype)

print("Extra =",extra)


bad_records = df.filter(

col("patient_id").isNull()

|

col("patient_name").isNull()

|

col("age").isNull()

|

(~col("gender").isin("M","F"))

|

col("admission_date").isNull()

)

bad_records.write \
.format("delta") \
.mode("overwrite") \
.save(

f"{quarantine_path}/patients"

)
display(bad_records)

good = df.subtract(

bad_records

)

good.write \
.format("delta") \
.mode("overwrite") \
.save(

f"{validated_path}/patients"

)
display(good)